# Setup — data for the session

**Run this notebook once, before the session.**

It downloads the dataset (~1.1 GB, peaks at ~3 GB RAM), subsets it to a single brain
section, applies basic QC and normalisation, and writes the result to
`data/brain_section.h5ad`. Notebooks 01–03 each load that file in a couple of seconds.

The data come from [*Molecularly defined and spatially resolved cell atlas of the whole
mouse brain* (Zhang et al., Nature, 2023)](https://www.nature.com/articles/s41586-023-06808-9);
[WB_MERFISH_animal2_coronal](https://cellxgene.cziscience.com/collections/0cca8620-8dee-45d0-aef5-23f032a5cf09).
The dataset was generated using **MERFISH**, which enables highly multiplexed
single-molecule RNA imaging with subcellular spatial resolution. The study profiles more than
**1,100** genes across roughly **10 million cells** from several adult **mouse brains**; the
object used here is one animal's coronal series (~1.9 M cells). The LIANA+ loader for it is
named `li.ds.yao_2023()`.

In [ ]:
import os
import scanpy as sc

import liana as li

# cache the download under ../../../../datasets/Hands_on_2_LIANA_MistYsets/Hands_on_2_LIANA_MistY rather than the current working directory
sc.settings.datasetdir = os.path.join(os.environ.get("MOPITAS_DATA", "../../../datasets"), "Hands_on_2_LIANA_MistY")

## Download and subset

The full object is ~1.9 M cells. We work with a single coronal section throughout.

In [ ]:
adata = li.ds.yao_2023()

# Subset to one brain section
adata = adata[adata.obs["brain_section_label"] == "C57BL6J-2.039"]
# In the CELLxGENE release `.X` is already log-normalised; the integer counts are in `.raw`
adata = adata.raw.to_adata()
# The dataset indexes vars by Ensembl ID; switch to gene symbols for LR lookup
adata.var_names = adata.var["gene_name"].astype(str)
adata.var_names.name = None
print(adata)

## QC and normalisation

The CELLxGENE object stores already log-normalised values in `.X` and the integer counts in
`.raw`, so above we started from `.raw`. Normalising `.X` again would log-transform twice.
Raw counts are kept in a layer; all methods below use normalised log1p values.

In [ ]:
# filter cells and genes
sc.pp.filter_cells(adata, min_genes=10)
sc.pp.filter_genes(adata, min_cells=3)

adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

print(adata.shape, "| max count:", adata.layers["counts"].max())

Spatial coordinates are in **µm** (MERFISH physical units), stored in `adata.obsm['X_spatial_coords']`.

In [ ]:
sc.pl.embedding(
    adata, basis="X_spatial_coords",
    color="cell_type", s=3, title="MERFISH — cell types"
)

## Ligand–receptor resource

LIANA+ ships a mouse-specific consensus resource, so no ortholog conversion is needed
here. (For organisms without a ready-made resource, `li.rs.get_hcop_orthologs` +
`li.rs.translate_resource` map the human `consensus` resource across.)

In [ ]:
resource = li.rs.select_resource('mouseconsensus')
print(f"{len(resource)} LR pairs in resource")
resource.head(5)

## Save

Everything downstream starts from this file.

In [ ]:
import os
adata.write_h5ad(os.path.join(os.environ.get("MOPITAS_DATA", "../../../datasets"), "Hands_on_2_LIANA_MistY", "brain_section.h5ad"))
print("wrote ../../../datasets/Hands_on_2_LIANA_MistY/brain_section.h5ad")

---

You are set up. In the session we will work through:

1. `01_restricting_lr_methods.ipynb` — which cell types talk to which, if they are close enough to?
2. `02_spatial_scales.ipynb` — at what distance does an interaction happen?
3. `03_local_hotspots.ipynb` — where in the tissue does it happen?